In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M25.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.770417427333544, 'n_it': 0.39735297757974103}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[15.200191199426692, 18.205536925165628, 17.699699459311336, 13.805736187738674, 13.24010576614382, 15.971940622808892, 15.031291990572326, 13.739670124437305, 16.40964106359555, 15.516610067991104, 13.869543184306155, 13.656570080060845, 15.615858913989644, 18.128389801805163, 15.683424710012304, 14.498127415994915, 13.515118814989716, 17.632399590834787, 15.352039877128192, 13.915623720293286, 13.914319052262327, 17.60151116808521, 14.535995425845583, 15.100851800934743, 15.380373415372214, 15.178803340219044, 15.085288598219424, 16.65794820294201, 15.990264525220033, 13.649494555475359, 14.48586271490677, 15.732763954253654, 14.257406900173411, 17.064169834874203, 13.794656879024703, 14.630707012027532, 14.463607947804226, 15.21814182824382, 14.78234530952668, 17.37733028593432, 15.601390299776194, 15.047990395728343, 13.815354552670192, 13.441988601270278, 14.59148092879603, 17.06550408681901, 14.049378427790193, 13.898904630673863, 15.213222146739035, 13.822592200260258, 15.291988

In [5]:
np.average(y_max_arr)

np.float64(15.062335110146366)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)